# ExgalM5WithCuts Healpix maps: impact of the dust-footprint threshold E(B-V)

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-19
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study + DESC static-probes metrics. The goal of the series `10_DESCMAFDEPTHANDGALCOUNT` is to understand in detail three `rubin_sim` MAF metrics (`ExgalM5WithCuts`, `GalaxyCountsMetricExtended`, `DepthLimitedNumGalMetric`) in order to improve them later (PSF handling, weak-lensing metric).
- This notebook: **01** of the series - the **`ExgalM5WithCuts`** metric (usable extragalactic i-band depth) recomputed on each dust-threshold footprint variant of the v5.3.6 simulations: Healpix maps, histograms, and maps of the differences between consecutive thresholds.
- Companion notebook: `02_compareGalaxyCounts.ipynb` (`GalaxyCountsMetricExtended` and `DepthLimitedNumGalMetric`). Source code of the three metrics: `00_DepthsAndCountsMetrics.ipynb`.
- Presentation model: `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb`
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_11yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**What `ExgalM5WithCuts` computes.** It is a per-pixel (`HealpixSlicer`) metric that returns the coadded, dust-corrected 5-sigma depth in one band (`lsst_filter`, default `i`), but **only for pixels that pass three quality cuts**; every other pixel is *masked*. In the order of `ExgalM5WithCuts.run`:

1. **Dust cut**: the pixel is masked if `E(B-V) > extinction_cut` (the `E(B-V)` value comes from the `DustMap` attached to the slicer).
2. **Filter-coverage cut**: the pixel is masked if the visits falling in it cover fewer than `n_filters` distinct bands. This is why the SQL query must **not** contain any band constraint.
3. **Depth**: keep only the `lsst_filter` visits and compute the coadded depth with `ExgalM5`: `m5_coadd = 1.25 * log10( sum_i 10**(0.8 * m5_i) )`, then subtract the band extinction `A_band = ax1[band] * E(B-V)`.
4. **Depth cut**: the pixel is masked if the resulting depth is below `depth_cut`.

This is the *parent* metric of the DESC 3x2pt FoM emulator (see `../06_MAF_DESC_TaskF/01_3x2pts_DESC_TaskForce_demo.ipynb`) and it is also the definition of the extragalactic footprint used inside `DepthLimitedNumGalMetric` (see notebook 02).

**Choices made in this notebook (all editable in Section 2).**
- The metric-side dust cut is kept **fixed at `E(B-V) < 0.2`** for the 7 runs, although each run was *scheduled* with a different threshold. Differences between the maps therefore come from the visit distribution produced by the scheduler for each footprint, not from a change in the metric.
- `depth_cut = 25.9` (year-10 value used by the official `science_radar_batch`: 26.0 minus an offset of 0.1), `n_filters = 6`, band `i`, `nside = 64`.
- All runs are truncated to their **first 10 years** (`night <= 10*365.25 + 0.5`), non-DDF visits only (`scheduler_note not like 'DD%'`). This matters because the baseline file is named `baseline_v5.3.6_11yrs`: without the truncation it would be deeper than the 10-year runs for a trivial reason.

**How the figures are built.**
1. Healpix map of the metric for each of the 7 runs, on one common color scale.
2. Area-weighted histograms of the valid pixels, and the cumulative area deeper than a given depth.
3. Difference maps between **consecutive** thresholds (`0.080-0.050`, `0.120-0.080`, `0.150-0.120`, `0.199-0.150`, `baseline-0.199`, `0.250-baseline`). A difference of depths only makes sense where **both** maps are valid, so the difference maps are restricted to the common footprint (subtracting a masked pixel as if it were 0 would create artificial steps of about 26 mag). The pixels **gained or lost** by the footprint are shown separately, in their own map and in the summary table.
4. A post-hoc study of the depth cut: since `ExgalM5WithCuts` returns the same depth whatever `depth_cut` is, a stricter cut (e.g. 26.0, the one used inside `DepthLimitedNumGalMetric`) is obtained exactly by masking the pixels below it, without rerunning MAF.

**Caching.** MAF is run only once per simulation; the resulting map is saved as `.npz` in `data_01_EXGALM5CUTS/` and reloaded at the next execution (set `FORCE_RECOMPUTE = True` to redo it).


## 1. Imports

In [ ]:
import os
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "EXGALM5CUTS"
data_dir = f"data_01_{NB_TAG}"
figs_dir = f"figs_01_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_11yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)

# Consecutive-threshold pairs (later run minus earlier run):
# 0.080-0.050, 0.120-0.080, 0.150-0.120, 0.199-0.150, baseline-0.199, 0.250-baseline
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]


def get_db_path(run_name):
    # Look in OPSIM_DIR, then in OPSIM_DIR/sim_baseline (where the baseline was stored in the 06/07 notebooks)
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached map, see section 5)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
BAND = "i"
N_FILTERS = 6  # number of bands (ugrizy) that must have at least one visit in the pixel
LIM_EBV = 0.2  # E(B-V) cut applied by the METRIC (identical for all runs, see overview)
DEPTH_CUT = 25.9  # i-band coadded depth cut (year-10 value of science_radar_batch: 26.0 - 0.1)
NSIDE = 64  # same nside as science_radar_batch and as the 07_variateEVmV notebooks
MAX_NIGHT = 10 * 365.25 + 0.5  # keep the first 10 years of every run

# No band constraint: ExgalM5WithCuts needs the visits of all bands to test the filter coverage
SQL_CONSTRAINT = "night <= %s and scheduler_note not like 'DD%%'" % MAX_NIGHT
INFO_LABEL = "ExgalM5WithCuts 10yr nonDD"
CONFIG_TAG = f"{BAND}_nf{N_FILTERS}_ebv{LIM_EBV:.2f}_depth{DEPTH_CUT:.2f}_nside{NSIDE}"
FORCE_RECOMPUTE = False  # True -> ignore the cached .npz maps and rerun MAF

pix_area = hp.nside2pixarea(NSIDE, degrees=True)
print(f"nside={NSIDE} -> npix={hp.nside2npix(NSIDE)}, pixel area = {pix_area:.4f} deg^2")
print("SQL constraint:", SQL_CONSTRAINT)
print("Cache tag     :", CONFIG_TAG)

## 3. The metric (docstring and source, to keep the calculation traceable)

In [ ]:
print(inspect.getdoc(metrics.ExgalM5WithCuts))
print("-" * 80)
print(inspect.getsource(metrics.ExgalM5WithCuts.run))
print("-" * 80)
print(inspect.getsource(ExgalM5.run))

## 4. Helper functions

Maps are handled as numpy masked arrays (masked = pixel rejected by the metric). `healpy` ignores the mask of a masked array, so masked pixels are converted to `hp.UNSEEN` (drawn in grey) before plotting (`as_healpy`).

In [ ]:
def as_healpy(m):
    # masked array -> plain array with hp.UNSEEN in the masked pixels (drawn in grey by healpy)
    return np.ma.filled(m, hp.UNSEEN)


def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def bundle_to_masked(bundle):
    # MetricBundle.metric_values -> float masked array (MAF badval, NaN and inf are all masked)
    data = np.ma.getdata(bundle.metric_values).astype(float)
    mask = np.ma.getmaskarray(bundle.metric_values) | ~np.isfinite(data) | (data < -600.0)
    return np.ma.masked_array(data, mask=mask)


def save_map(path, m):
    np.savez_compressed(path, data=np.ma.getdata(m), mask=np.ma.getmaskarray(m))


def load_map(path):
    with np.load(path) as f:
        return np.ma.masked_array(f["data"], mask=f["mask"])


def diff_map(map_b, map_a, fill_masked=None):
    # Pixel-by-pixel map_b - map_a.
    # fill_masked=None: keep the difference only where BOTH maps are valid (masked elsewhere).
    # fill_masked=0.0 : a masked pixel counts as 0 (e.g. no galaxy counted there); the result is
    #                   masked only where BOTH maps are masked.
    mask_b, mask_a = np.ma.getmaskarray(map_b), np.ma.getmaskarray(map_a)
    b = np.ma.getdata(map_b).astype(float)
    a = np.ma.getdata(map_a).astype(float)
    if fill_masked is None:
        return np.ma.masked_array(b - a, mask=mask_b | mask_a)
    b = np.where(mask_b, fill_masked, b)
    a = np.where(mask_a, fill_masked, a)
    return np.ma.masked_array(b - a, mask=mask_b & mask_a)


def pair_stats(map_b, map_a, dmap, with_totals=False):
    # Footprint bookkeeping and statistics of one difference map
    vb, va = ~np.ma.getmaskarray(map_b), ~np.ma.getmaskarray(map_a)
    common, gained, lost = vb & va, vb & ~va, va & ~vb
    d = dmap.compressed()
    row = {
        "area_a_deg2": va.sum() * pix_area,
        "area_b_deg2": vb.sum() * pix_area,
        "area_gained_deg2": gained.sum() * pix_area,
        "area_lost_deg2": lost.sum() * pix_area,
        "diff_mean": d.mean() if d.size else np.nan,
        "diff_median": np.median(d) if d.size else np.nan,
        "diff_std": d.std() if d.size else np.nan,
        "diff_min": d.min() if d.size else np.nan,
        "diff_max": d.max() if d.size else np.nan,
    }
    if with_totals:
        b = np.ma.getdata(map_b).astype(float)
        a = np.ma.getdata(map_a).astype(float)
        row.update(
            {
                "total_a": a[va].sum(),
                "total_b": b[vb].sum(),
                "total_diff": b[vb].sum() - a[va].sum(),
                "from_common_pixels": (b[common] - a[common]).sum(),
                "from_gained_pixels": b[gained].sum(),
                "from_lost_pixels": -a[lost].sum(),
            }
        )
    return row


def per_run_summary(maps, with_sum=False):
    rows = []
    for run_name, dust in RUNS_INFO:
        v = maps[run_name].compressed()
        rows.append(
            {
                "run": run_name,
                "E(B-V) cut of the run": dust,
                "n_pixels": v.size,
                "area_deg2": v.size * pix_area,
                "mean": v.mean() if v.size else np.nan,
                "median": np.median(v) if v.size else np.nan,
                "std": v.std() if v.size else np.nan,
                "min": v.min() if v.size else np.nan,
                "max": v.max() if v.size else np.nan,
            }
        )
        if with_sum:
            rows[-1]["sum"] = v.sum()
    return pd.DataFrame(rows).set_index("run")


def plot_all_maps(maps, tag, unit, title, cmap="viridis"):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
        hp.mollview(
            as_healpy(maps[run_name]),
            fig=fig.number,
            sub=(3, 3, i),
            min=vmin,
            max=vmax,
            cmap=cmap,
            title=f"{run_name}\n{run_label(run_name)}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(f"{title} (common color scale: 1st-99th percentile of all runs)", fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_healpix_allruns")
    plt.show()


def plot_overlay_hist(maps, tag, xlabel, title, nbins=80):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    bins = np.linspace(np.percentile(pooled, 0.1), np.percentile(pooled, 99.9), nbins + 1)
    colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    for (run_name, dust), c in zip(RUNS_INFO, colors):
        v = maps[run_name].compressed()
        w = np.full(v.size, pix_area)
        axs[0].hist(v, bins=bins, weights=w, histtype="step", color=c, label=run_label(run_name))
        axs[1].hist(
            v, bins=bins, weights=w, histtype="step", color=c, cumulative=-1, label=run_label(run_name)
        )
    axs[0].set_ylabel("Area per bin [deg²]")
    axs[1].set_ylabel("Area with value ≥ x [deg²]")
    for ax in axs:
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, f"{tag}_histograms_allruns")
    plt.show()


def plot_diff_map(dmap, run_b, run_a, tag, unit, label):
    vlim = max(np.percentile(np.abs(dmap.compressed()), 99), 1e-6) if dmap.count() else 1.0
    fig = plt.figure(figsize=(8, 5))
    hp.mollview(
        as_healpy(dmap),
        fig=fig.number,
        min=-vlim,
        max=vlim,
        cmap="RdBu_r",
        title=f"{label} difference: {run_b}\nminus {run_a}",
        unit=unit,
    )
    hp.graticule()
    save_fig(fig, f"{tag}_diff_" + f"{run_b}_MINUS_{run_a}".replace(".", "_"))
    plt.show()
    return vlim


def plot_diff_mosaic(diff_maps, tag, unit, label):
    vlim = max(max(np.percentile(np.abs(d.compressed()), 99) for d in diff_maps.values() if d.count()), 1e-6)
    fig = plt.figure(figsize=(16, 10))
    for i, ((run_b, run_a), d) in enumerate(diff_maps.items(), start=1):
        hp.mollview(
            as_healpy(d),
            fig=fig.number,
            sub=(2, 3, i),
            min=-vlim,
            max=vlim,
            cmap="RdBu_r",
            title=f"{run_b}\nminus {run_a}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(
        f"{label}: differences between consecutive dust thresholds (common scale ±{vlim:.3g})",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_diff_allpairs_combined")
    plt.show()


def plot_footprint_change(maps, tag, label):
    cmap = ListedColormap(["tab:red", "lightgreen", "tab:blue"])
    fig = plt.figure(figsize=(16, 10))
    for i, (run_b, run_a) in enumerate(PAIRS, start=1):
        vb, va = ~np.ma.getmaskarray(maps[run_b]), ~np.ma.getmaskarray(maps[run_a])
        status = np.zeros(vb.size)
        status[vb & ~va] = 1.0
        status[va & ~vb] = -1.0
        status = np.ma.masked_array(status, mask=~(va | vb))
        hp.mollview(
            as_healpy(status),
            fig=fig.number,
            sub=(2, 3, i),
            min=-1,
            max=1,
            cmap=cmap,
            cbar=False,
            title=f"{run_b}\nminus {run_a}",
        )
    fig.suptitle(
        f"{label}: footprint change between consecutive dust thresholds "
        "(blue = pixel gained, red = pixel lost, green = valid in both, dark grey = valid in neither)",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_footprint_change_combined")
    plt.show()


def plot_diff_histograms(diff_maps, tag, xlabel, label):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, ((run_b, run_a), d) in zip(axes.flat, diff_maps.items()):
        ax.hist(d.compressed(), bins=100, color="steelblue")
        ax.axvline(0, color="k", linewidth=0.8)
        ax.set_title(f"{run_b}\n- {run_a}", fontsize=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N pixels")
    fig.suptitle(f"{label}: histograms of the consecutive differences")
    fig.tight_layout()
    save_fig(fig, f"{tag}_diff_histograms")
    plt.show()


def analyse_pairs(maps, tag, label, unit, xlabel, fill_masked=None, with_totals=False):
    # Difference maps (individual + mosaic), footprint change, histograms and table for all PAIRS
    diff_maps, rows = {}, []
    for run_b, run_a in PAIRS:
        print(f"=== {run_b}  -  {run_a} ===")
        d = diff_map(maps[run_b], maps[run_a], fill_masked=fill_masked)
        diff_maps[(run_b, run_a)] = d
        plot_diff_map(d, run_b, run_a, tag, unit, label)
        row = {"pair (E(B-V) thresholds)": f"{DUST_THRESH[run_b]:.3f} - {DUST_THRESH[run_a]:.3f}"}
        row.update(pair_stats(maps[run_b], maps[run_a], d, with_totals=with_totals))
        rows.append(row)
    plot_diff_mosaic(diff_maps, tag, unit, label)
    plot_footprint_change(maps, tag, label)
    plot_diff_histograms(diff_maps, tag, xlabel, label)
    stats = pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")
    stats.to_csv(join(data_dir, f"{tag}_diff_summary.csv"))
    print("Saved:", join(data_dir, f"{tag}_diff_summary.csv"))
    return diff_maps, stats

## 5. Compute (or reload) the `ExgalM5WithCuts` map of each run

The first execution runs MAF on the 7 databases (this can take several minutes per run); the maps are then cached in `data_01_EXGALM5CUTS/`.

In [ ]:
dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def cache_path(run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"ExgalM5WithCuts_{tag}_{CONFIG_TAG}.npz")


def load_or_run(run_name):
    path = cache_path(run_name)
    if isfile(path) and not FORCE_RECOMPUTE:
        print(f"[{run_name}] loaded cached map: {path}")
        return load_map(path)

    dbpath = get_db_path(run_name)
    print(f"[{run_name}] running ExgalM5WithCuts on {dbpath}")
    metric = metrics.ExgalM5WithCuts(
        lsst_filter=BAND, n_filters=N_FILTERS, extinction_cut=LIM_EBV, depth_cut=DEPTH_CUT
    )
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    bundle = mb.MetricBundle(
        metric, slicer, SQL_CONSTRAINT, maps_list=[dustmap], run_name=run_name, info_label=INFO_LABEL
    )
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list([bundle]), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()
    m = bundle_to_masked(bundle)
    save_map(path, m)
    return m


depth_maps = {run_name: load_or_run(run_name) for run_name, _ in RUNS_INFO}

## 6. Per-run summary

`area_deg2` is the area of the pixels that pass the three cuts (the usable extragalactic footprint); `mean`/`median`/`std`/`min`/`max` describe the coadded i-band depth over these pixels.

In [ ]:
summary_df = per_run_summary(depth_maps)
summary_csv = join(data_dir, "ExgalM5WithCuts_per_run_summary.csv")
summary_df.to_csv(summary_csv)
print("Saved:", summary_csv)
summary_df.round(3)

## 7. Healpix maps of the usable i-band depth for the 7 footprint variants

Grey pixels are masked: they fail the dust cut, the 6-band coverage cut, or the depth cut.

In [ ]:
plot_all_maps(
    depth_maps,
    "ExgalM5WithCuts",
    "i-band depth [mag]",
    "ExgalM5WithCuts: coadded dust-corrected i-band depth after cuts",
)

## 8. Histograms

Left: area-weighted histogram of the depth (each pixel weighs its area). Right: area deeper than a given depth (reverse cumulative), the natural way to compare footprints.

In [ ]:
plot_overlay_hist(
    depth_maps,
    "ExgalM5WithCuts",
    "i-band coadded depth [mag]",
    "ExgalM5WithCuts: area-weighted histograms (left) and area deeper than x (right)",
)

## 9. Differences between consecutive dust thresholds

For each pair, the map is `depth(run_b) - depth(run_a)` with `run_b` the run with the *larger* threshold. It is computed on the pixels valid in **both** runs; a positive value means that `run_b` is deeper there. Each individual map has its own symmetric color scale (99th percentile of `|difference|`), the mosaic uses one common scale so that the six pairs can be compared.

The footprint mosaic and the table give the pixels **gained** (valid in `run_b` only) and **lost** (valid in `run_a` only). Note that `0.199` and `baseline (0.200)` have almost the same nominal footprint: the `baseline-0.199` pair is a near-null test whose differences give the scale of the scheduler-to-scheduler fluctuations, to be compared with the other pairs.

In [ ]:
diff_maps, stats = analyse_pairs(
    depth_maps,
    "ExgalM5WithCuts",
    "ExgalM5WithCuts",
    "Δ depth [mag]",
    "Δ depth [mag] (pixels valid in both runs)",
)

In [ ]:
stats.round(3)

## 10. Sensitivity to the depth cut (post-hoc, no MAF rerun)

`ExgalM5WithCuts` returns the same depth for every pixel that survives, whatever `depth_cut` is. A **stricter** cut is therefore obtained exactly by masking the pixels below it. This is used to see what the `depth_cut = 26.0` hard-wired in `DepthLimitedNumGalMetric` (`lim_mag_i_ptsrc`) does to the footprint, compared with the `25.9` used above for the 3x2pt-like configuration.

In [ ]:
POSTHOC_CUTS = [25.9, 26.0, 26.1, 26.2]
assert min(POSTHOC_CUTS) >= DEPTH_CUT, "a post-hoc cut can only be stricter than the cut used in the MAF run"

rows = []
for run_name, dust in RUNS_INFO:
    v = depth_maps[run_name].compressed()
    row = {"run": run_name, "E(B-V) cut of the run": dust}
    for cut in POSTHOC_CUTS:
        row[f"area, depth >= {cut:.1f} [deg2]"] = np.sum(v >= cut) * pix_area
    rows.append(row)
posthoc_df = pd.DataFrame(rows).set_index("run")
posthoc_csv = join(data_dir, "ExgalM5WithCuts_area_vs_depthcut.csv")
posthoc_df.to_csv(posthoc_csv)
print("Saved:", posthoc_csv)

fig, ax = plt.subplots(figsize=(7, 5))
for cut in POSTHOC_CUTS:
    ax.plot(
        posthoc_df["E(B-V) cut of the run"],
        posthoc_df[f"area, depth >= {cut:.1f} [deg2]"],
        marker="o",
        label=f"depth >= {cut:.1f}",
    )
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("Usable area [deg²]")
ax.set_title("ExgalM5WithCuts: usable area vs footprint threshold")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
save_fig(fig, "ExgalM5WithCuts_area_vs_dust_threshold")
plt.show()

posthoc_df.round(3)

## 11. Notes for a PSF-aware version of the metric

What the code above does and does not know about the PSF:

- The per-visit column `fiveSigmaDepth` is a **point-source** depth computed by the simulator (sky brightness, airmass, exposure time and the seeing of that visit). The PSF therefore enters `ExgalM5WithCuts` **only implicitly**, through this column, and the coadd (`1.25 log10 sum 10**(0.8 m5)`) simply adds these point-source depths.
- `depth_cut` is a fixed number of magnitudes: it does not depend on the seeing of the visits that build the pixel, and no galaxy size or resolution criterion enters the cuts.
- `ExgalM5WithCuts.run` receives the visits of **all** bands in `data_slice`, and the columns it reads are declared in its `col=[...]` list: other OpSim columns (for instance the effective seeing `seeingFwhmEff`) can be added there and used in `run`, which is the natural hook for a seeing-aware depth.
- Per-run seeing maps for these same simulations already exist in `../07_variateEVmV/01_Seeing_HealpixMaps.ipynb`.

Notebook 02 shows where the PSF (or rather its absence) matters for the galaxy-count metrics.

## Caveats

- The metric-side cut `E(B-V) < 0.2` is the same for all runs. Pixels with `0.2 < E(B-V) < 0.25` are therefore masked even in the `0.250` run, and for the runs scheduled with a threshold below 0.2 the pixels between the run threshold and 0.2 are masked only if they fail the coverage/depth cuts because they were not (or barely) observed.
- Difference maps are restricted to pixels valid in both runs. The pixels gained or lost by the footprint are reported separately (footprint-change mosaic and `area_gained_deg2` / `area_lost_deg2` in the table).
- Two different scheduler runs do not observe the same visit sequence even where their footprint is identical, so part of the pixel-to-pixel differences is scheduling noise (see the `baseline-0.199` pair).
- The `nside = 64` pixel is 0.84 deg² (same as the official batch); the effective areas are quantized by this pixel size.
- The 10-year truncation applies to every run, so the numbers are not those of the full `baseline_v5.3.6_11yrs` simulation.


## References
- `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py` (`ExgalM5WithCuts`), `rubin_sim/maf/metrics/exgal_m5.py` (`ExgalM5`): https://github.com/lsst/rubin_sim/tree/main/rubin_sim/maf/metrics
- `rubin_sim.maf.batches.science_radar_batch` (official depth cuts per year): https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/batches/science_radar_batch.py
- `../06_MAF_DESC_TaskF/01_3x2pts_DESC_TaskForce_demo.ipynb` - the 3x2pt chain built on `ExgalM5WithCuts`.
- `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb` - presentation model (maps, consecutive differences, histograms).
- Lochner, M. et al. 2018, arXiv:1808.00006, "Optimizing LSST Observing Strategy for Dark Energy Science"
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/
